**Github repository:** https://github.com/NicholasBorch/ComSocSci-Assignments

**Group Members:** Nicholas Borch (s234841) and Robin Braagaard (s234856)

**Distribution of Contribution:** All group members contributed equally in the planning and execution of solutions and implementation of code for the assignment tasks.

---

# 02467 Computational social science - Assignment 2

----

### Python libraries used in assignment

In [1]:
### Importing libraries
import pandas as pd
import numpy as np
from pathlib import Path
import networkx as nx
from collections import defaultdict, Counter
from itertools import combinations
import json
import random
import copy
import matplotlib.pyplot as plt
from tqdm import tqdm 
import scipy.stats as stats
from joblib import Parallel, delayed
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
from typing import Dict, Any, List, Tuple
from tqdm.auto import tqdm
from contextlib import contextmanager
import re
import community.community_louvain as community_louvain
import os
import warnings
import netwulf as nw
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore", category=FutureWarning)

---

## Pre Assignment 2 dataset preparation:

Before starting assignment 2, we need to complete and create the final datasets necessary for the following tasks (Authors, Papers, Abstracts). Our goal is to collect co-author data (D4, D5, D6) and combine with IC2S2 data (D1, D2, D3). The datasets are discussed in week 3. We have our input datasets (from Assignment 1) D1 (Authors.csv (IC2S2 venue authors)), D2 (Papers.csv (IC2S2 venue papers)), and D3 (Abstracts.jsonl (IC2S2 venue abstracts)). What we need to collect before assignment 2 is then D4 (Co-authors of IC2S2 authors), D5 (Papers co-authored by IC2S2 + co-authors), and D6 (Abstracts of D5 papers). This results in the final datasets being Authors.csv (D1 + D4 merged), Papers.csv (D2 + D5 merged, and keeping collaborative papers only), and Abstracts.jsonl (D3 + D6 merged).

In [ ]:
# Configuration of setup

A1_DATA_DIR = Path("A1_data")
OUT_DIR = Path("A2_data")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Reusing API credentials from Assignment 1
API_CREDENTIALS = [
    {"key": "OrnWTgBF4MDTq5Lj41EJoP", "email": "s234841@dtu.dk"},
    {"key": "5tTODa6zVIcgByCyqvxdrf", "email": "rbraagaard@gmail.com"},
    {"key": "zfogWMSQFJEZVu7vfoKtRJ", "email": "robinfifa0099@gmail.com"},
    {"key": "CvsqIRMWLKZ4XjnFzuhHKr", "email": "robinburner0099@gmail.com"},
    {"key": "y9jNIDioxpDJFcjmDdAjJm", "email": "s234856@dtu.dk"},
]

CURRENT_KEY_INDEX = 0

# Keeping track of API keys and usage
def get_current_api_key() -> str:
    return API_CREDENTIALS[CURRENT_KEY_INDEX]["key"]

def get_current_headers() -> Dict[str, str]:
    email = API_CREDENTIALS[CURRENT_KEY_INDEX]["email"]
    return {"User-Agent": f"IC2S2 Assignment (mailto:{email})"}

def rotate_api_key():
    global CURRENT_KEY_INDEX
    CURRENT_KEY_INDEX += 1
    if CURRENT_KEY_INDEX >= len(API_CREDENTIALS):
        raise RuntimeError("All API keys exhausted.")
    print(f"Switched to API key index {CURRENT_KEY_INDEX}")


# Defining OpenAlex endpoints
OPENALEX_BASE = "https://api.openalex.org"
WORKS_ENDPOINT = f"{OPENALEX_BASE}/works"
AUTHORS_ENDPOINT = f"{OPENALEX_BASE}/authors"
CONCEPTS_ENDPOINT = f"{OPENALEX_BASE}/concepts"

# Setting request parameters
PER_PAGE = 200
AUTHOR_BATCH_SIZE = 25
COAUTHOR_META_BATCH_SIZE = 50
N_JOBS = 6
REQUEST_TIMEOUT = 30
MIN_SLEEP_BETWEEN_REQUESTS_SEC = 0.05

In [ ]:
# Helper functions for API calls

def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=6,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("https://", adapter)
    s.headers.update(get_current_headers())
    return s

def openalex_get(session: requests.Session, url: str, params: Dict[str, Any]) -> Dict[str, Any]:
    global CURRENT_KEY_INDEX
    params = dict(params)

    while True:
        params["api_key"] = get_current_api_key()
        session.headers.update(get_current_headers())

        r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)

        if r.status_code == 429 and "Insufficient budget" in r.text:
            rotate_api_key()
            session.headers.update(get_current_headers())
            continue

        if r.status_code == 429:
            time.sleep(1.5)
            continue

        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code}: {r.text[:300]}")

        time.sleep(MIN_SLEEP_BETWEEN_REQUESTS_SEC)
        return r.json()

def author_id_short(openalex_author_id_url: str) -> str:
    if not isinstance(openalex_author_id_url, str):
        return ""
    return openalex_author_id_url.rstrip("/").split("/")[-1]

def work_id_short(openalex_work_id_url: str) -> str:
    if not isinstance(openalex_work_id_url, str):
        return ""
    return openalex_work_id_url.rstrip("/").split("/")[-1]

def chunk_list(xs: List[str], chunk_size: int) -> List[List[str]]:
    return [xs[i:i + chunk_size] for i in range(0, len(xs), chunk_size)]

@contextmanager
def tqdm_joblib(tqdm_object):
    import joblib
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

In [ ]:
# Loading D1, D2, D3 from Assignment 1

print("="*60)
print("Loading Assignment 1 datasets (D1, D2, D3)")
print("="*60)

D1_PATH = A1_DATA_DIR / "D1_openalex_authors.csv"
D2_PATH = A1_DATA_DIR / "D2_ic2s2_papers.csv"
D3_PATH = A1_DATA_DIR / "D3_ic2s2_abstracts.jsonl"

df_d1 = pd.read_csv(D1_PATH)
df_d2 = pd.read_csv(D2_PATH)

# Creating sets meant for filtering
ic2s2_author_ids = set(author_id_short(x) for x in df_d1["id"].dropna().tolist())
ic2s2_author_ids = {a for a in ic2s2_author_ids if a.startswith("A")}

ic2s2_work_ids = set(df_d2["id"].dropna().astype(str).tolist())
ic2s2_work_ids = {w for w in ic2s2_work_ids if w.startswith("W")}

print(f"IC2S2 authors (D1): {len(ic2s2_author_ids):,}")
print(f"IC2S2 works (D2): {len(ic2s2_work_ids):,}")
print(f"D3 exists: {D3_PATH.exists()}")

In [ ]:
# Extracting co-author IDs from D2 (excluding IC2S2 authors)

print("\n" + "="*60)
print("Extracting co-author IDs from D2")
print("="*60)

all_author_ids = (df_d2["author_ids"].fillna("").astype(str).str.split("|").explode().str.strip())

unique_coauthor_ids = set([a for a in all_author_ids.tolist() if a and a.startswith("A")])
coauthor_ids_only = unique_coauthor_ids - ic2s2_author_ids

print(f"Total unique authors in D2: {len(unique_coauthor_ids):,}")
print(f"Co-authors (excluding IC2S2 authors): {len(coauthor_ids_only):,}")

In [ ]:
# Collecting D4 (Co-author metadata)

print("\n" + "="*60)
print("Collecting D4 (Co-author metadata from OpenAlex)")
print("="*60)

D4_PATH = OUT_DIR / "D4_ic2s2_coauthors.csv"

def fetch_coauthors_metadata_batch(author_ids_batch: List[str]) -> List[Dict[str, Any]]:
    """Fetching author metadata for a batch of author IDs for efficiency"""
    s = make_session()
    authors_or = "|".join(author_ids_batch)
    
    data = openalex_get(s, AUTHORS_ENDPOINT,params={"filter": f"openalex_id:{authors_or}", "per-page": COAUTHOR_META_BATCH_SIZE})
    
    results = data.get("results", []) or []
    rows = []
    
    for author in results:
        # Setting filters from week 3, where only works with works_count between 5 and 5000 are included
        works_count = author.get("works_count", 0)
        if not (5 <= works_count <= 5000):
            continue
            
        # Extracting last known institution country code
        country_code = None
        lkis = author.get("last_known_institutions")
        if isinstance(lkis, list) and lkis and isinstance(lkis[0], dict):
            country_code = lkis[0].get("country_code")
        
        rows.append({
            "id": author.get("id"),
            "id_short": author_id_short(author.get("id")),
            "display_name": author.get("display_name"),
            "works_api_url": author.get("works_api_url"),
            "h_index": (author.get("summary_stats") or {}).get("h_index"),
            "works_count": works_count,
            "country_code": country_code,
        })
    
    return rows

# Batch co-authors and fetch
coauthor_ids_list = list(coauthor_ids_only)
coauthor_batches = chunk_list(coauthor_ids_list, COAUTHOR_META_BATCH_SIZE)

print(f"Fetching metadata for {len(coauthor_ids_list):,} co-authors in {len(coauthor_batches):,} batches...")

pbar = tqdm(total=len(coauthor_batches), desc="Fetching co-author metadata")

with tqdm_joblib(pbar):
    results = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(fetch_coauthors_metadata_batch)(batch) for batch in coauthor_batches)

all_d4_rows = []
for batch_rows in results:
    all_d4_rows.extend(batch_rows)

df_d4 = pd.DataFrame(all_d4_rows).drop_duplicates(subset=["id_short"]).reset_index(drop=True)
df_d4.to_csv(D4_PATH, index=False)

print(f"D4 saved: {D4_PATH}")
print(f"Co-authors with 5-5000 works: {len(df_d4):,}")

# Updating the valid co-author set (after filtering)
valid_coauthor_ids = set(df_d4["id_short"].tolist())
all_network_author_ids = ic2s2_author_ids | valid_coauthor_ids

In [ ]:
# Collecting concept IDs for filtering

print("\n" + "="*60)
print("Looking up CSS and Quant concept IDs")
print("="*60)

def get_level0_concept_id(session: requests.Session, concept_name: str) -> str:
    data = openalex_get(session, CONCEPTS_ENDPOINT, params={"search": concept_name, "per-page": 25})
    results = data.get("results", []) or []
    level0 = [c for c in results if c.get("level") == 0]
    target = concept_name.strip().lower()

    for c in level0:
        if str(c.get("display_name", "")).strip().lower() == target:
            return str(c["id"]).rstrip("/").split("/")[-1]
    return str(level0[0]["id"]).rstrip("/").split("/")[-1]

session = make_session()

css_names = ["Sociology", "Psychology", "Economics", "Political science"]
quant_names = ["Mathematics", "Physics", "Computer science"]

CSS_IDS = [get_level0_concept_id(session, n) for n in css_names]
QUANT_IDS = [get_level0_concept_id(session, n) for n in quant_names]

print(f"CSS concepts: {dict(zip(css_names, CSS_IDS))}")
print(f"Quant concepts: {dict(zip(quant_names, QUANT_IDS))}")

In [ ]:
# Collecting D5 (Co-author papers) and D6 (Co-author abstracts)

print("\n" + "="*60)
print("Collecting D5 (Co-author papers) and D6 (Co-author abstracts)")
print("="*60)

D5_PATH = OUT_DIR / "D5_coauthors_papers.csv"
D6_PATH = OUT_DIR / "D6_coauthors_abstracts.jsonl"

def build_works_filter(author_ids_batch: List[str]) -> str:
    authors_or = "|".join(author_ids_batch)
    css_or = "|".join(CSS_IDS)
    quant_or = "|".join(QUANT_IDS)

    # Setting filters defined by week 3 requirements
    parts = [
        f"authorships.author.id:{authors_or}",
        "cited_by_count:>10",
        "authors_count:<10",
        f"concept.id:{css_or}",
        f"concept.id:{quant_or}",
    ]
    return ",".join(parts)

def extract_d5_d6_from_work(w: Dict[str, Any], valid_network_ids: set) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    wid = work_id_short(w.get("id"))

    # Extracting author_ids and filterering to only network authors
    a_ids: List[str] = []
    for a in (w.get("authorships") or []):
        aid_full = (a.get("author") or {}).get("id")
        if isinstance(aid_full, str) and aid_full:
            aid = author_id_short(aid_full)
            if aid in valid_network_ids:  # Only keep network authors
                a_ids.append(aid)

    d5 = {
        "id": wid,
        "publication_year": w.get("publication_year"),
        "cited_by_count": w.get("cited_by_count"),
        "author_ids": "|".join([x for x in a_ids if x]),
    }

    d6 = {
        "id": wid,
        "title": w.get("title") if isinstance(w.get("title"), str) else "",
        "abstract_inverted_index": w.get("abstract_inverted_index"),
    }

    return d5, d6

def fetch_works_for_coauthor_batch(author_ids_batch: List[str]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    s = make_session()
    filt = build_works_filter(author_ids_batch)

    cursor = "*"
    d5_rows: List[Dict[str, Any]] = []
    d6_rows: List[Dict[str, Any]] = []

    while True:
        data = openalex_get(s, WORKS_ENDPOINT, params={"filter": filt, "per-page": PER_PAGE, "cursor": cursor})
        results = data.get("results", []) or []

        for w in results:
            wid = work_id_short(w.get("id"))
            
            # Skipping if already in D2 (IC2S2 papers)
            if wid in ic2s2_work_ids:
                continue
                
            d5, d6 = extract_d5_d6_from_work(w, all_network_author_ids)
            if d5["id"]:
                d5_rows.append(d5)
                d6_rows.append(d6)

        next_cursor = (data.get("meta") or {}).get("next_cursor")
        if not next_cursor:
            break
        cursor = next_cursor

    return d5_rows, d6_rows

# Batching co-authors for works collection
valid_coauthor_ids_list = list(valid_coauthor_ids)
coauthor_works_batches = chunk_list(valid_coauthor_ids_list, AUTHOR_BATCH_SIZE)

print(f"Fetching works for {len(valid_coauthor_ids_list):,} co-authors in {len(coauthor_works_batches):,} batches...")

pbar = tqdm(total=len(coauthor_works_batches), desc="Fetching co-author works")

with tqdm_joblib(pbar):
    results = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(fetch_works_for_coauthor_batch)(batch) for batch in coauthor_works_batches)

all_d5_rows = []
all_d6_rows = []

for d5_rows, d6_rows in results:
    all_d5_rows.extend(d5_rows)
    all_d6_rows.extend(d6_rows)

df_d5 = pd.DataFrame(all_d5_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
df_d6 = pd.DataFrame(all_d6_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)

# Saving datasets D5 and D6
df_d5.to_csv(D5_PATH, index=False)

with open(D6_PATH, "w", encoding="utf-8") as f:
    for _, row in df_d6.iterrows():
        f.write(json.dumps({
                "id": row["id"],
                "title": row["title"],
                "abstract_inverted_index": row["abstract_inverted_index"],
            },
            ensure_ascii=False) + "\n")

print(f"D5 saved: {D5_PATH} ({len(df_d5):,} papers)")
print(f"D6 saved: {D6_PATH} ({len(df_d6):,} abstracts)")

In [ ]:
# Creating the final datasets (Authors, Papers, Abstracts)

print("\n" + "="*60)
print("Creating the final datasets (Authors, Papers, Abstracts)")
print("="*60)


## Final Authors = D1 + D4

# Standardizing D1 to match the D4 structure
df_d1_std = pd.DataFrame({
    "id": df_d1["id"],
    "id_short": df_d1["id"].apply(author_id_short),
    "display_name": df_d1["display_name"],
    "works_api_url": df_d1["works_api_url"],
    "h_index": df_d1["h_index"],
    "works_count": df_d1["works_count"],
    "country_code": df_d1["country_code"],
}).dropna(subset=["id_short"]).copy()

Authors = pd.concat([df_d1_std, df_d4], ignore_index=True)
Authors = Authors.drop_duplicates(subset=["id_short"]).reset_index(drop=True)

AUTHORS_OUT = OUT_DIR / "Authors.csv"
Authors.to_csv(AUTHORS_OUT, index=False)
print(f"Completed dataset Authors.csv: {len(Authors):,} authors - {AUTHORS_OUT}")


## Final Papers = D2 + D5 (collaborative works only)

Papers = pd.concat([df_d2, df_d5], ignore_index=True)
Papers = Papers.drop_duplicates(subset=["id"]).reset_index(drop=True)

# Keeping only collaborative works (meaning 2+ network authors)
def count_pipe_ids(s):
    if not isinstance(s, str) or not s.strip():
        return 0
    return len([x for x in s.split("|") if x.strip()])

Papers["n_authors_network"] = Papers["author_ids"].apply(count_pipe_ids)
Papers = Papers[Papers["n_authors_network"] >= 2].copy()
Papers = Papers.drop(columns=["n_authors_network"]).reset_index(drop=True)

PAPERS_OUT = OUT_DIR / "Papers.csv"
Papers.to_csv(PAPERS_OUT, index=False)
print(f"Completed dataset Papers.csv: {len(Papers):,} papers (collaborative only) - {PAPERS_OUT}")


## Final Abstracts = D3 + D6

ABSTRACTS_OUT = OUT_DIR / "Abstracts.jsonl"

paper_ids = set(Papers["id"].dropna().astype(str).tolist())
written_ids = set()

print(f"Writing Abstracts (filtered to {len(paper_ids):,} final papers)...")

with open(D3_PATH, "r", encoding="utf-8", buffering=1024*1024) as fin, \
     open(ABSTRACTS_OUT, "w", encoding="utf-8", buffering=1024*1024) as fout:

    # Stream D3
    for line in tqdm(fin, desc="Scanning D3"):
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        wid = obj.get("id")
        if not wid or wid not in paper_ids:
            continue
        if wid in written_ids:
            continue
        written_ids.add(wid)
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

    # Append D6
    for _, row in tqdm(df_d6.iterrows(), total=len(df_d6), desc="Scanning D6"):
        wid = row.get("id")
        if not wid or wid not in paper_ids:
            continue
        if wid in written_ids:
            continue
        written_ids.add(wid)

        rec = {
            "id": wid,
            "title": row.get("title", ""),
            "abstract_inverted_index": row.get("abstract_inverted_index", None),
        }
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")

missing_abs = len(paper_ids - written_ids)

print(f"Completed dataset Abstracts.jsonl: {len(written_ids):,} abstracts - {ABSTRACTS_OUT}")
print(f"Papers missing abstracts: {missing_abs}")

In [ ]:
# Here is a final summary of the datasets for assignment 2

print("\n" + "="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"Authors.csv:     {len(Authors):,} researchers")
print(f"Papers.csv:      {len(Papers):,} collaborative papers")
print(f"Abstracts.jsonl: {len(written_ids):,} abstracts")
print("="*60)